In [1]:
import aiohttp
import asyncio
import pandas as pd

In [2]:
subreddits = ["finance","stocks","investing","wallstreetbets","economics","cryptocurrency","options","stockmarket","valueinvesting","financialindependence"]

In [10]:
async def fetch(session, url):
    async with session.get(url) as resp:
        if resp.status == 429:
            print("Rate limited. Waiting...")
            await asyncio.sleep(30)
            print("Rate limited. Retrying...")
            return await fetch(session, url)

        return await resp.json()

async def get_comments(session, post_id):
    await asyncio.sleep(1)
    url = f"https://api.reddit.com/comments/{post_id}?limit=100"
    data = await fetch(session, url)
    comments = []

    for c in data[1]["data"]["children"]:
        if c["kind"] == "t1":
            comments.append(c["data"]["body"])

    return comments

async def main(subreddits: str, limit: int = 100):
    reddit_list = []
    HEADERS = {"User-Agent": "python:reddit.scraper:v1.0 (by /u/test)"}

    async with aiohttp.ClientSession(headers=HEADERS) as session:

        after = None
        collected = 0

        while collected < limit:
            url = f"https://api.reddit.com/r/{subreddits}/new?limit=100"
            if after:
                url += f"&after={after}"
            posts_data = await fetch(session, url)
            posts = posts_data["data"]["children"]
            after = posts_data["data"]["after"]

            for p in posts:
                post = p["data"]
                title = post["title"]
                post_id = post["id"]
                num_comments = post["num_comments"]
                comments = await get_comments(session, post_id)
                reddit_list.append({"subreddit": subreddits,"title": title,"num_comments": num_comments,"comments": comments})

                collected += 1

                if collected >= limit:
                    break

            if after is None:
                break

            return reddit_list


In [11]:
reddit_list = []
for s in subreddits:
    s=await main(s,1000)
    reddit_list.extend(s)
    await asyncio.sleep(30)

In [12]:
reddit_list

[{'subreddit': 'finance',
  'title': 'Gulf Bonds Safe haven status: under fire everywhere but credit markets',
  'num_comments': 0,
  'comments': []},
 {'subreddit': 'finance',
  'title': 'US intervention in oil futures would be ‘biblical disaster’, CME warns. Terry Duffy says any attempt by the government to lower prices using derivatives market would erode confidence',
  'num_comments': 23,
  'comments': ['Oh they’re definitely going to do it then. ',
   'A terrible idea and guaranteed disaster? Put in on the schedule.',
   '&gt;The head of CME Group has warned the Trump administration it risks a “biblical disaster” if it attempts to lower oil prices by intervening in derivatives markets during the war with Iran. \n\n&gt;Terry Duffy, the chief executive of CME Group, which runs the exchange where US oil futures trade, told a conference this week that it would erode market confidence if the US government stepped into the futures market in a bid to curb the rise in crude. \n\n&gt;“Mark

In [13]:
df=pd.DataFrame(reddit_list)

In [14]:
df.head()

,subreddit,title,num_comments,comments
0,finance,Gulf Bonds Safe haven status: under fire every...,0,[]
1,finance,US intervention in oil futures would be ‘bibli...,23,"[Oh they’re definitely going to do it then. , ..."
2,finance,A Guide to the Fault Lines in the Credit Market,3,[Everyone’s in debt and no way to pay it back ...
3,finance,British fintech Revolut gets full banking licence,37,[More Lithuanian than British. But they want t...
4,finance,The Iran conflict is delivering four commodity...,60,"[Helium should be added to the list too., Mont..."


In [15]:
df.shape

(1000, 4)

In [16]:
df.value_counts('subreddit')

subreddit
cryptocurrency           100
economics                100
finance                  100
financialindependence    100
investing                100
options                  100
stockmarket              100
stocks                   100
valueinvesting           100
wallstreetbets           100
Name: count, dtype: int64

In [17]:
df.to_csv(r"C:\Users\Asus\PycharmProjects\test_2\smth.csv")

In [11]:
import nltk
from mpmath.libmp import round_up
from nltk.sentiment import SentimentIntensityAnalyzer
from tqdm.notebook import tqdm
from transformers import AutoTokenizer,AutoModelForSequenceClassification,pipeline
from scipy.special import softmax

In [12]:
vad=SentimentIntensityAnalyzer()

In [17]:
pipe=pipeline("sentiment-analysis",device=0)
res=pipe(df.iloc[:,4].tolist(),batch_size=64,truncation=True)

IndexError: single positional indexer is out-of-bounds